# 07 Near-Duplicate, Spam, dan Bot

Menemukan teks salinan dan akun yang mencurigakan. Contoh: komentar TikTok tentang MBG. Notebook ini jalan di CPU.

**Daftar isi**

1. Kartu Masalah
2. Kapan Pakai Notebook Ini
3. Config
4. EDA
5. Preprocessing
6. Validasi
7. Baseline
8. Model Utama
9. Post-processing
10. Post-analysis
11. Keluaran

## 1. Kartu Masalah

| Bagian | Isi |
|---|---|
| **Task** | Menemukan komentar kembar, spam, dan akun mencurigakan |
| **Unit prediksi** | Pasangan komentar, lalu grup, lalu akun |
| **Metrik** | Recall dan presisi pada duplikat sintetis |
| **Masalah terbesar** | Kembar sering sudah diubah sedikit: pencocokan persis hanya menangkap 48,85% salinan. Banyak juga komentar pendek yang sama secara wajar, seperti emoji. |
| **Data contoh** | 5.000 komentar TikTok MBG, nama akun disamarkan |

## 2. Kapan Pakai Notebook Ini

**Pakai kalau** soal meminta mendeteksi buzzer, salinan komentar, atau spam. Juga dipakai sebagai langkah awal
di task lain, karena duplikat bisa membocorkan data antar fold.

**Yang perlu diganti kalau datanya beda:**
- `load_data`, `TEXT_COL`, `USER_COL`.
- `SPAM_PATTERNS`.
- Tanpa kolom pengguna, hapus bagian skor akun di bagian 10.

## 3. Config

**Import dan lingkungan.** Mode offline dinyalakan di sini, dan progress bar serta warning
dimatikan supaya output tetap bersih.

In [1]:
import importlib.util
import json
import os
import random
import subprocess
import sys
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

os.environ.update(
    CUDA_VISIBLE_DEVICES="0",
    HF_HUB_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
    TOKENIZERS_PARALLELISM="false",
    TQDM_DISABLE="1",
    PYTHONWARNINGS="ignore",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)
warnings.filterwarnings("ignore")
sys.modules["tensorflow"] = None  # tidak dipakai; mencegah log TensorFlow saat umap diimpor

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 90, "display.max_rows", 30, "display.width", 140)
plt.rcParams.update(
    {
        "figure.figsize": (8, 3.5),
        "figure.dpi": 90,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print(f"device={DEVICE} | torch={torch.__version__}")

device=cpu | torch=2.11.0+cpu


**Lokasi file.** `find_pack` mencari paket offline, dan `find_file` mencari file data di Kaggle
atau folder lokal. Di mesin lokal, set `PACK_DIR` dan `DATA_ROOT` lewat environment variable.

In [2]:
INPUT_ROOTS = [Path(p) for p in (os.environ.get("DATA_ROOT"), "/kaggle/input", ".") if p]


def find_pack() -> Path:
    """Cari folder paket offline lewat file pack_manifest.json."""
    if os.environ.get("PACK_DIR"):
        return Path(os.environ["PACK_DIR"])
    for root in INPUT_ROOTS:
        for manifest in root.glob("**/pack_manifest.json"):
            return manifest.parent
    raise FileNotFoundError("Paket offline tidak ketemu. Set environment variable PACK_DIR.")


def find_file(name: str) -> Path:
    """Cari file data berdasarkan nama di semua folder input."""
    for root in INPUT_ROOTS:
        for path in root.glob(f"**/{name}"):
            return path
    raise FileNotFoundError(f"File {name} tidak ketemu di {INPUT_ROOTS}")


def install_offline(*modules: str) -> None:
    """Pasang library dari wheel paket offline, hanya bila belum terpasang."""
    missing = [m for m in modules if importlib.util.find_spec(m.replace("-", "_")) is None]
    if missing:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-index",
                "--find-links",
                str(PACK_DIR / "wheels"),
                "--root-user-action=ignore",
                *missing,
            ],
            check=True,
        )


PACK_DIR = find_pack()
MODEL_DIR = PACK_DIR / "models"
EXTRA_DATA_DIR = PACK_DIR / "data"
RESULT_DIR = Path("hasil")
RESULT_DIR.mkdir(exist_ok=True)
print(f"paket offline: {PACK_DIR}")

paket offline: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline


**Pencatat waktu dan ringkasan.** `stage` mencatat durasi tiap tahap, dan `run_summary` mencetak
ringkasan akhir yang dibaca skrip audit.

In [3]:
RUN_START = time.time()
STAGE_SECONDS: dict[str, float] = {}


@contextmanager
def stage(name: str):
    """Catat durasi satu tahap."""
    start = time.time()
    yield
    STAGE_SECONDS[name] = round(time.time() - start, 1)
    print(f"[waktu] {name}: {STAGE_SECONDS[name]:.0f} detik")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_summary(task: str, n_rows: int, metrics: dict, needs_gpu: bool = True) -> None:
    """Simpan dan cetak ringkasan run."""
    total = int(time.time() - RUN_START)
    used_gpu = torch.cuda.is_available() and torch.cuda.max_memory_allocated() > 0
    summary = {
        "task": task,
        "mode": MODE,
        "use_pretrained": USE_PRETRAINED,
        "n_rows": n_rows,
        "total_detik": total,
        "gpu": used_gpu,
        "tahap_detik": STAGE_SECONDS,
        "metrik": metrics,
    }
    (RESULT_DIR / "ringkasan.json").write_text(json.dumps(summary, indent=1))
    note = "" if needs_gpu else " gpu_tidak_wajib"
    print(
        f"RINGKASAN_RUN task={task} mode={MODE} n_rows={n_rows} total_detik={total} "
        f"gpu={used_gpu}{note}"
    )
    print(json.dumps(metrics, indent=1))

**Pengaturan utama.** Semua yang perlu diubah ada di sel ini.

In [4]:
MODE = "penuh"  # "cepat" (<30 menit) atau "penuh" (<2 jam)
USE_PRETRAINED = False  # notebook ini tidak butuh model dari luar
SEED = 42
N_ROWS = 5000
TEXT_COL, USER_COL = "comment", "username"
SPAM_PATTERNS = r"http|www\.|wa\.me|whatsapp|promo|diskon|slot|gacor|cek bio|klik link"
INJECT_SHARE = 0.1

SETTINGS = {
    "cepat": {"thresholds": [0.7, 0.8, 0.9], "eval_seeds": [1]},
    "penuh": {"thresholds": [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95], "eval_seeds": [1, 2, 3]},
}[MODE]

seed_everything(SEED)
print(SETTINGS)

{'thresholds': [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95], 'eval_seeds': [1, 2, 3]}


**Util teks.** `clean_text` hanya membuang gangguan teknis (URL, mention, spasi) dan
mempertahankan emoji serta tanda seru karena keduanya membawa sinyal. `normalize_slang` memakai
kamus alay. `near_duplicate_groups` menemukan komentar kembar supaya tidak bocor antar fold.

In [5]:
import re  # noqa: E402

from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import roc_auc_score  # noqa: E402
from sklearn.model_selection import cross_val_predict  # noqa: E402
from sklearn.neighbors import NearestNeighbors  # noqa: E402

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Bersihkan URL, mention, dan spasi berlebih. Huruf kecil, emoji tetap."""
    text = URL_RE.sub(" ", str(text))
    text = MENTION_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip().lower()


def load_slang_dict() -> dict[str, str]:
    """Muat kamus kata informal ke kata baku."""
    path = next((EXTRA_DATA_DIR / "kamus_alay").glob("**/colloquial-indonesian-lexicon.csv"))
    lexicon = pd.read_csv(path)
    return dict(zip(lexicon["slang"].astype(str), lexicon["formal"].astype(str)))


def normalize_slang(text: str, slang: dict[str, str]) -> str:
    """Ganti kata informal dengan kata baku, per kata."""
    return " ".join(slang.get(word, word) for word in text.split())


def near_duplicate_groups(texts: list[str], threshold: float = 0.9) -> np.ndarray:
    """Beri id grup yang sama pada teks yang kemiripan char TF-IDF-nya di atas threshold."""
    vectors = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(texts)
    neighbors = NearestNeighbors(n_neighbors=min(6, len(texts)), metric="cosine").fit(vectors)
    distance, index = neighbors.kneighbors(vectors)
    parent = list(range(len(texts)))

    def root(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, (row_dist, row_idx) in enumerate(zip(distance, index)):
        for dist, j in zip(row_dist, row_idx):
            if dist <= 1 - threshold:
                parent[root(i)] = root(int(j))
    return np.array([root(i) for i in range(len(texts))])


def adversarial_auc(texts_a: list[str], texts_b: list[str]) -> float:
    """AUC model yang membedakan dua kumpulan teks. Dekat 0,5 berarti tidak ada shift."""
    texts = list(texts_a) + list(texts_b)
    target = np.array([0] * len(texts_a) + [1] * len(texts_b))
    vectors = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    scores = cross_val_predict(model, vectors, target, cv=3, method="predict_proba")[:, 1]
    return float(roc_auc_score(target, scores))


def text_profile(texts: pd.Series) -> pd.Series:
    """Ringkasan kualitas teks untuk EDA."""
    words = texts.astype(str).str.split().str.len()
    return pd.Series(
        {
            "baris": len(texts),
            "kosong": int((words == 0).sum()),
            "duplikat_persis_%": round(100 * texts.duplicated().mean(), 1),
            "kata_median": float(words.median()),
            "kata_p95": float(words.quantile(0.95)),
            "kata_maks": int(words.max()),
            "ada_url_%": round(100 * texts.astype(str).str.contains(r"https?://").mean(), 1),
            "ada_emoji_%": round(
                100 * texts.astype(str).str.contains(r"[\U0001F300-\U0001FAFF]").mean(), 1
            ),
        }
    )

## 4. EDA

Muat data, lalu lihat komentar kembar persis dan akun yang paling aktif.

In [6]:
def load_data() -> pd.DataFrame:
    frame = pd.read_excel(find_file("mbg_tiktokcomment.xlsx"))
    frame = frame.dropna(subset=[TEXT_COL])
    frame[TEXT_COL] = frame[TEXT_COL].astype(str)
    frame = frame.sample(n=min(N_ROWS, len(frame)), random_state=SEED).reset_index(drop=True)
    frame[USER_COL] = "akun_" + pd.Series(pd.factorize(frame[USER_COL])[0]).astype(
        str
    )  # samarkan nama akun
    return frame


data = load_data()
data["clean"] = data[TEXT_COL].map(clean_text)
print(f"baris={len(data)} | akun unik={data[USER_COL].nunique()}")
print("akun teraktif:", data[USER_COL].value_counts().head(5).to_dict())
print("komentar berulang:", data["clean"].value_counts().head(5).to_dict())
print("cocok pola spam:", round(100 * data["clean"].str.contains(SPAM_PATTERNS).mean(), 2), "%")
text_profile(data[TEXT_COL])

baris=5000 | akun unik=4727
akun teraktif: {'akun_909': 6, 'akun_387': 5, 'akun_930': 5, 'akun_1011': 4, 'akun_794': 3}
komentar berulang: {'': 39, '[sticker]': 36, '🥰🥰🥰': 21, '🥰': 19, '😁😁😁': 14}
cocok pola spam: 0.18 %


baris                5000.0
kosong                  0.0
duplikat_persis_%       7.8
kata_median             7.0
kata_p95               33.0
kata_maks             284.0
ada_url_%               0.0
ada_emoji_%            36.9
dtype: float64

**Temuan dan keputusan**

| Temuan | Keputusan |
|---|---|
| 7,8% duplikat persis, 36,9% berisi emoji | Komentar yang terlalu pendek tidak dicek, sisa 4.349 baris |
| Pola spam hanya 0,18% | Spam kata kunci bukan masalah utama, fokus ke duplikat |
| Akun unik 4.727 dari 5.000 baris | Skor akun lemah di data ini, karena hampir semua akun hanya berkomentar sekali |

## 5. Preprocessing

Dua kunci pencocokan: `kunci` hanya huruf dan angka untuk kembar persis, dan char n-gram untuk kembar yang
sedikit diubah. Karakter berulang dirapikan, jadi `bagusss` sama dengan `bagus`.

In [7]:
def match_key(text: str) -> str:
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    return re.sub(r"[^a-z0-9]", "", text)


data["kunci"] = data["clean"].map(match_key)
usable = data[data["kunci"].str.len() >= 8].reset_index(drop=True)
print(f"baris cukup panjang untuk dicek: {len(usable)} dari {len(data)}")

baris cukup panjang untuk dicek: 4349 dari 5000


## 6. Validasi

Tidak ada label duplikat, jadi dibuat **injeksi**: sebagian komentar disalin lalu diubah sedikit. Metriknya
recall (salinan yang ketemu) dan presisi pasangan (pasangan yang dilaporkan memang salinan).

In [8]:
def mutate(text: str, rng: np.random.Generator) -> str:
    """Salin dengan perubahan kecil: hapus huruf, tukar kata, atau tambah emoji dan tanda."""
    words = text.split()
    choice = rng.integers(0, 3)
    if choice == 0 and len(words) > 2:
        i = rng.integers(0, len(words) - 1)
        words[i], words[i + 1] = words[i + 1], words[i]
    elif choice == 1:
        i = rng.integers(0, len(words))
        if len(words[i]) > 3:
            words[i] = words[i][:-1]
    else:
        words.append(str(rng.choice(["!!", "😂", "..", "🙏"])))
    return " ".join(words)


def build_eval(seed: int) -> tuple[list[str], set[tuple[int, int]]]:
    rng = np.random.default_rng(seed)
    texts = usable["clean"].tolist()
    sources = rng.choice(len(texts), int(INJECT_SHARE * len(texts)), replace=False)
    truth = set()
    for source in sources:
        truth.add((int(source), len(texts)))
        texts.append(mutate(texts[source], rng))
    return texts, truth


def pair_scores(
    found: set[tuple[int, int]], truth: set[tuple[int, int]], n_original: int
) -> dict[str, float]:
    """Skor hanya memperhatikan pasangan yang melibatkan baris injeksi, karena hanya itu yang jawabannya diketahui."""
    relevant = {p for p in found if p[1] >= n_original}
    hits = relevant & truth
    return {
        "recall": round(len(hits) / max(len(truth), 1), 4),
        "presisi": round(len(hits) / max(len(relevant), 1), 4),
    }


assert pair_scores({(0, 5), (1, 2)}, {(0, 5), (3, 6)}, 5) == {"recall": 0.5, "presisi": 1.0}
results: list[dict] = []

## 7. Baseline

Kembar persis setelah dibersihkan. Tanpa kesalahan, tapi tidak menangkap salinan yang diubah.

In [9]:
def exact_pairs(texts: list[str]) -> set[tuple[int, int]]:
    seen: dict[str, int] = {}
    pairs = set()
    for i, text in enumerate(texts):
        key = match_key(text)
        if key in seen:
            pairs.add((seen[key], i))
        else:
            seen[key] = i
    return pairs


for seed in SETTINGS["eval_seeds"]:
    texts, truth = build_eval(seed)
    results.append(
        {
            "metode": "kembar_persis",
            "seed": seed,
            **pair_scores(exact_pairs(texts), truth, len(usable)),
        }
    )
pd.DataFrame(results)

,metode,seed,recall,presisi
0,kembar_persis,1,0.5046,0.9563
1,kembar_persis,2,0.4700,0.9273
2,kembar_persis,3,0.4908,0.9682


## 8. Model Utama

Char TF-IDF + tetangga terdekat. Char n-gram tahan terhadap salah ketik dan perubahan kecil.

In [10]:
def neighbor_table(texts: list[str], n_neighbors: int = 6):
    vectors = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True
    ).fit_transform(texts)
    model = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", n_jobs=-1).fit(vectors)
    return model.kneighbors(vectors)


def char_pairs(distance, index, threshold: float) -> set[tuple[int, int]]:
    pairs = set()
    for i, (row_distance, row_index) in enumerate(zip(distance, index)):
        for dist, j in zip(row_distance, row_index):
            if int(j) != i and dist <= 1 - threshold:
                pairs.add((min(i, int(j)), max(i, int(j))))
    return pairs


with stage("tetangga"):
    for seed in SETTINGS["eval_seeds"]:
        texts, truth = build_eval(seed)
        distance, index = neighbor_table(texts)
        for threshold in SETTINGS["thresholds"]:
            found = char_pairs(distance, index, threshold)
            results.append(
                {
                    "metode": f"char_tfidf_{threshold}",
                    "seed": seed,
                    **pair_scores(found, truth, len(usable)),
                }
            )
result_table = pd.DataFrame(results).groupby("metode")[["recall", "presisi"]].mean().round(4)
result_table["f1"] = (
    2
    * result_table["recall"]
    * result_table["presisi"]
    / (result_table["recall"] + result_table["presisi"]).clip(lower=1e-9)
).round(4)
result_table.sort_values("f1", ascending=False)

[waktu] tetangga: 5 detik


,recall,presisi,f1
metode,,,
char_tfidf_0.8,0.9854,0.9058,0.9439
char_tfidf_0.85,0.9708,0.9103,0.9396
char_tfidf_0.75,0.9931,0.8905,0.9390
char_tfidf_0.9,0.9585,0.9173,0.9374
char_tfidf_0.7,0.9985,0.8689,0.9292
char_tfidf_0.65,0.9992,0.8503,0.9188
char_tfidf_0.95,0.8940,0.9202,0.9069
char_tfidf_0.6,1.0000,0.8154,0.8983
kembar_persis,0.4885,0.9506,0.6454


## 9. Post-processing

Pilih threshold dengan F1 tertinggi dari injeksi, terapkan ke data asli, lalu satukan pasangan menjadi grup
dengan union-find. Satu grup berarti satu kampanye teks yang mungkin sama.

In [11]:
result_table.to_csv(RESULT_DIR / "perbandingan.csv")
candidates = result_table[result_table.index.str.startswith("char")]
best_threshold = float(candidates["f1"].idxmax().rsplit("_", 1)[1])
print("threshold terpilih:", best_threshold)
usable["grup"] = near_duplicate_groups(usable["clean"].tolist(), threshold=best_threshold)
group_sizes = usable["grup"].value_counts()
print(
    f"grup berisi lebih dari satu: {(group_sizes > 1).sum()} | baris di dalamnya: {group_sizes[group_sizes > 1].sum()}"
)

threshold terpilih: 0.8


grup berisi lebih dari satu: 185 | baris di dalamnya: 397


## 10. Post-analysis

**Grup terbesar.** Baca isinya: komentar umum seperti `mantap` bukan bukti kampanye.

In [12]:
rows = []
for group, size in group_sizes.head(10).items():
    members = usable[usable["grup"] == group]
    rows.append(
        {
            "grup": int(group),
            "jumlah": int(size),
            "akun_unik": members[USER_COL].nunique(),
            "contoh": members["clean"].iloc[0][:80],
        }
    )
pd.DataFrame(rows)

,grup,jumlah,akun_unik,contoh
0,1346,10,10,alhamdulillah mbg di smp ku
1,3149,5,5,mbg hari ini ~
2,4333,5,5,mbg makan gratis beracun
3,3546,4,3,saya setuju dengan ibuk ini 👍
4,3524,4,3,"jo: ibuk ngomongnya, times new roman, bold, caplock nyala, 16, ctrl +e"
5,3849,4,3,bubarin bubarin
6,3010,4,1,pendukung fanatik prabowo anomali 78 ini udah ga ketolong yaa😭 bisa2 nya ada ora
7,2685,3,3,mantap bu
8,3487,3,3,"adalah burgerrrrrrrrrrrrrr benda tipis berwana pink'""💕"""""
9,4283,3,3,bener banget bu


**Skor kecurigaan akun.** Gabungan aktivitas, porsi komentar duplikat, dan porsi pola spam. Nama akun sudah
disamarkan. Ini daftar untuk diperiksa, bukan vonis akun tersebut bot.

In [13]:
usable["duplikat"] = usable["grup"].map(group_sizes) > 1
usable["spam"] = usable["clean"].str.contains(SPAM_PATTERNS)
account = usable.groupby(USER_COL).agg(
    komentar=("clean", "size"),
    porsi_duplikat=("duplikat", "mean"),
    porsi_spam=("spam", "mean"),
    video_unik=("video_id", "nunique"),
)
account = account[account["komentar"] >= 2]
account["skor"] = (
    account[["komentar", "porsi_duplikat", "porsi_spam"]].rank(pct=True).mean(axis=1).round(3)
)
account.sort_values("skor", ascending=False).head(8).round(3)

,komentar,porsi_duplikat,porsi_spam,video_unik,skor
username,,,,,
akun_909,6,0.667,0.0,1,0.581
akun_794,3,0.667,0.0,1,0.568
akun_930,5,0.000,0.0,1,0.538
akun_387,4,0.000,0.0,1,0.536
akun_1010,2,1.000,0.0,1,0.532
akun_1015,2,1.000,0.0,1,0.532
akun_1045,2,1.000,0.0,1,0.532
akun_1227,2,1.000,0.0,1,0.532


## 11. Keluaran

File untuk dikumpulkan dan ringkasan run.

In [14]:
usable[["grup", "duplikat", "spam"]].rename_axis("id").to_csv(RESULT_DIR / "submission.csv")
best_row = result_table.loc[candidates["f1"].idxmax()].to_dict()
run_summary(
    "duplikat_spam", len(data), {"threshold": best_threshold, "injeksi": best_row}, needs_gpu=False
)

RINGKASAN_RUN task=duplikat_spam mode=penuh n_rows=5000 total_detik=17 gpu=False gpu_tidak_wajib
{
 "threshold": 0.8,
 "injeksi": {
  "recall": 0.9854,
  "presisi": 0.9058,
  "f1": 0.9439
 }
}


**Poin untuk presentasi**

- Char TF-IDF dengan threshold 0,8: F1 0,9439, sedangkan kembar persis 0,6454. Berjalan 17 detik di CPU.
- Ditemukan 185 grup berisi 397 baris.
- Grup terbesar seperti `alhamdulillah mbg di smp ku` muncul dari 10 akun berbeda. Ini indikasi, bukan bukti bot.
- Skor sintetis hanya berlaku untuk perubahan yang kita simulasikan, jadi periksa contoh grup secara manual.